In [ ]:
!date

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from scipy.sparse import csr_matrix, hstack, vstack
from scipy.io import mmread, mmwrite

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/processed/rna_finegrain.h5ad')

In [ ]:
adata

In [ ]:
adata.obs

In [ ]:
rna_clusters_path = f'{projdir}/csv/label_transfer/snmct_xgboost_v7.csv'
rna_clusters = pd.read_csv(rna_clusters_path, sep='\t', index_col=0)
rna_clusters.shape

In [ ]:
rna_clusters.head()

In [ ]:
mcg_clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/integrate_m3c/leiden_clusters_renamed.csv'
mcg_clusters = pd.read_csv(mcg_clusters_path, sep='\t', index_col=0)
mcg_clusters.shape

In [ ]:
mcg_clusters.head()

In [ ]:
tmp_meta = adata.obs.copy()
tmp_meta['mcg_leiden'] = mcg_clusters.reindex(tmp_meta.index)['renamed_leiden'].copy()
tmp_meta['rna_cluster'] = rna_clusters.reindex(tmp_meta.index)['pred_coarse'].copy()
tmp_meta['time'] = rna_clusters.reindex(tmp_meta.index)['time_num'].copy()

tmp_meta = tmp_meta.dropna()
tmp_meta.shape

In [ ]:
tmp_meta.head()

In [ ]:
donors = ['C29', 'C37', 'C38', 'C39']
tmp_results = []

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
x_lab = 'mcg_leiden'
y_lab = 'value'
hue = 'variable'

for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    donor_meta = tmp_meta[tmp_meta['line'] == d].copy()
    donor_meta['mcg_leiden'] = donor_meta['mcg_leiden'].replace('Inter_2', 'Fibro_15')
    time_meta = donor_meta[donor_meta['time'] == 0]
    
    mcg_plot = time_meta.melt(id_vars=['mcg_leiden'], value_vars=['S.Score', 'G2M.Score'])
    
    sns.boxplot(data=mcg_plot, x=x_lab, y=y_lab, hue=hue, ax=ax, showfliers=False)
    ax.set_xlabel('', fontsize=16)
    ax.set_ylabel('', fontsize=16)
    ax.set_title(f'{d}', fontsize=20)

    ax.tick_params(axis='x', labelsize=9, labelrotation=45)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    mcg_plot['donor'] = d
    tmp_results.append(mcg_plot)
    
plt.subplots_adjust(wspace=0.4, hspace=0.4)

In [ ]:
merged_scores = pd.concat(tmp_results)
merged_scores.shape

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'
merged_scores.to_csv(f'{outdir}/mcg_leiden_fibro_snmct_cc_rna.csv', sep='\t')

In [ ]:
all_dists = []

for d in donors:
    donor_meta = tmp_meta[tmp_meta['line'] == d]    
    tmp_clusts = sorted(donor_meta['mcg_leiden'].unique())
    donor_dist = pd.DataFrame(index=tmp_meta['Phase'].unique())
    for s in tmp_clusts:
        leiden_meta = donor_meta[donor_meta['mcg_leiden'] == s]
        donor_dist[s] = leiden_meta['Phase'].value_counts()
    donor_dist = donor_dist.T
    all_dists.append(donor_dist)

In [ ]:
all_props = [x.div(x.sum(axis=1), axis=0) for x in all_dists]

In [ ]:
all_props[0]

In [ ]:
all_props[1]

In [ ]:
all_props

In [ ]:
tmp_results = []

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
x_lab = 'mcg_leiden'
y_lab = 'value'
hue = 'variable'

for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    donor_meta = all_props[i]
    donor_meta['mcg_leiden'] = donor_meta.index
    to_plot = donor_meta.melt(id_vars=['mcg_leiden'], value_vars=['G0/G1', 'S', 'G2M'])
    
    sns.barplot(data=to_plot, x=x_lab, y=y_lab, hue=hue, ax=ax)
    ax.set_xlabel('', fontsize=16)
    ax.set_ylabel('', fontsize=16)
    ax.set_title(f'{d}', fontsize=20)

    ax.tick_params(axis='x', labelsize=12, labelrotation=45)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    to_plot['donor'] = d
    tmp_results.append(to_plot)
    
plt.subplots_adjust(wspace=0.5, hspace=0.5)

In [ ]:
merged_props = pd.concat(tmp_results)
merged_props.shape

In [ ]:
merged_props.head()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'
merged_props.to_csv(f'{outdir}/epiclone_rna_cellcycle_prop.csv', sep='\t')

### let's do it de novo

In [ ]:
g2m_genes_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/csv/cell_cycle/g2m_genes.txt'
g2m_genes = pd.read_csv(g2m_genes_path, header=None)[0].values

s_genes_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/csv/cell_cycle/s_genes.txt'
s_genes = pd.read_csv(s_genes_path, header=None)[0].values

In [ ]:
ensg_path = '/u/project/cluo/chliu/Genomes/IGVF_hg38/annotations/ensembl_to_symbol.tsv'
ensg = pd.read_csv(ensg_path, sep='\t')
ensg_dict = dict(zip(ensg['gene'], ensg['symbol']))

In [ ]:
adata.var['ensg'] = adata.var.index
adata.var['symbol'] = [ensg_dict.get(x) for x in adata.var['ensg']]

In [ ]:
adata.var.index = adata.var['symbol']

In [ ]:
adata.var.head()

In [ ]:
mask = ~adata.var.index.duplicated()
tmp_adata = adata[:,mask]
tmp_adata

In [ ]:
sc.tl.score_genes_cell_cycle(tmp_adata, s_genes=s_genes, g2m_genes=g2m_genes)

In [ ]:
!date